# Data Gathering and Unified Dataset Creation


**Download and extract all required datasets using the Kaggle API**

**PlantVillage Dataset**: A diverse dataset containing multiple crops and their major diseases.

**Rice Disease Dataset**: Images of rice crops affected by different diseases, organized by class.

**Cotton Leaf Disease Dataset**: Contains images of cotton leaves categorized by disease type.

**Sugarcane Leaf Disease Dataset**: Images of sugarcane leaves grouped by disease type.

In [ ]:
# Download all required datasets using the Kaggle API
# Steps:
# 1. Upload kaggle.json (API key)
# 2. Configure path and permissions
# 3. Download listed datasets
# 4. Unzip and organize them into dataset folders

import os
from zipfile import ZipFile

# 1. Upload your kaggle.json
from google.colab import files
print("Upload kaggle.json from your Kaggle account...")
files.upload()

# 2. Move kaggle.json to the correct location and set permissions
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. List of datasets to download (added PlantVillage as the last one)
datasets = [
    "anshulm257/rice-disease-dataset",
    "seroshkarim/cotton-leaf-disease-dataset",
    "nirmalsankalana/sugarcane-leaf-disease-dataset",
    "abdallahalidev/plantvillage-dataset"  # Added PlantVillage dataset
]

# 4. Create a folder for datasets
base_dir = "/content/plant_datasets"
os.makedirs(base_dir, exist_ok=True)

# 5. Download and unzip datasets
for ds in datasets:
    print(f"\nDownloading: {ds}")
    !kaggle datasets download -d {ds} -p {base_dir}
    # Find the downloaded zip file(s)
    zip_files = [f for f in os.listdir(base_dir) if f.endswith('.zip')]
    for zipf in zip_files:
        zip_path = os.path.join(base_dir, zipf)
        extract_folder = os.path.join(base_dir, zipf.replace(".zip",""))
        os.makedirs(extract_folder, exist_ok=True)
        print(f"Extracting {zipf}...")
        with ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)
        os.remove(zip_path)
print("\nAll datasets have been downloaded and extracted to", base_dir)


Upload kaggle.json from your Kaggle account...


Saving kaggle.json to kaggle.json

Downloading: anshulm257/rice-disease-dataset
Dataset URL: https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset
License(s): unknown
 95% 968M/0.99G [00:04<00:00, 153MB/s]
100% 0.99G/0.99G [00:05<00:00, 210MB/s]
Extracting rice-disease-dataset.zip...

Downloading: seroshkarim/cotton-leaf-disease-dataset
Dataset URL: https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset
License(s): unknown
 66% 119M/181M [00:00<00:00, 1.24GB/s]
100% 181M/181M [00:00<00:00, 1.03GB/s]
Extracting cotton-leaf-disease-dataset.zip...

Downloading: nirmalsankalana/sugarcane-leaf-disease-dataset
Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/sugarcane-leaf-disease-dataset
License(s): CC0-1.0
 86% 138M/160M [00:00<00:00, 1.44GB/s]
100% 160M/160M [00:00<00:00, 1.23GB/s]
Extracting sugarcane-leaf-disease-dataset.zip...

Downloading: abdallahalidev/plantvillage-dataset
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage

**Class-wise image distribution for each dataset**

This step ensures we understand the dataset composition before merging.

In [ ]:
import os
from collections import Counter

base_dir = "/content/plant_datasets"
dataset_class_counts = {}

for dataset_name in os.listdir(base_dir):
    dataset_path = os.path.join(base_dir, dataset_name)
    if not os.path.isdir(dataset_path):
        continue

    class_counter = Counter()
    for root, dirs, files in os.walk(dataset_path):
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if image_files:
            class_name = os.path.basename(root)
            class_counter[class_name] += len(image_files)

    dataset_class_counts[dataset_name] = class_counter

# Print results
for dataset, classes in dataset_class_counts.items():
    print(f"\nDataset: {dataset}")
    for cls, count in classes.items():
        print(f"  {cls}: {count} images")



Dataset: plantvillage-dataset
  Pepper,_bell___Bacterial_spot: 2991 images
  Tomato___Leaf_Mold: 2856 images
  Blueberry___healthy: 4506 images
  Tomato___Spider_mites Two-spotted_spider_mite: 5028 images
  Tomato___Late_blight: 5727 images
  Cherry_(including_sour)___Powdery_mildew: 3156 images
  Pepper,_bell___healthy: 4434 images
  Grape___Esca_(Black_Measles): 4150 images
  Corn_(maize)___healthy: 3486 images
  Strawberry___Leaf_scorch: 3327 images
  Orange___Haunglongbing_(Citrus_greening): 16521 images
  Grape___Black_rot: 3540 images
  Apple___Black_rot: 1863 images
  Potato___Late_blight: 3000 images
  Strawberry___healthy: 1368 images
  Tomato___Early_blight: 3000 images
  Tomato___healthy: 4773 images
  Corn_(maize)___Northern_Leaf_Blight: 2955 images
  Cherry_(including_sour)___healthy: 2562 images
  Grape___healthy: 1269 images
  Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 3228 images
  Tomato___Tomato_mosaic_virus: 1119 images
  Squash___Powdery_mildew: 5505 images
  Corn

**Standardizing class labels and merging datasets**

Class names are reformatted to the structure **"CropName___DiseaseName"** to ensure consistency.
The datasets are then merged into a unified dataset.

In [ ]:
# Standardize and merge datasets:
# - Define mapping for rice, cotton, and sugarcane
# - Copy images into unified dataset folder
# - Skip grayscale/segmented folders
# - Track and summarize class counts

import os
import shutil
from collections import defaultdict

source_base = "/content/plant_datasets"
merged_base = "/content/merged_dataset/train"

os.makedirs(merged_base, exist_ok=True)

# Datasets to include
datasets_to_add = [
    "plantvillage-dataset",
    "rice-disease-dataset",
    "cotton-leaf-disease-dataset",
    "sugarcane-leaf-disease-dataset"
]

# Manual mapping for rice, cotton, and sugarcane
manual_map = {
    # Rice
    "Bacterial Leaf Blight": "Rice___Bacterial_leaf_blight",
    "Sheath Blight": "Rice___Sheath_blight",
    "Leaf scald": "Rice___Leaf_scald",
    "Healthy Rice Leaf": "Rice___healthy",
    "Leaf Blast": "Rice___Leaf_blast",
    "Brown Spot": "Rice___Brown_spot",

    # Cotton
    "bacterial_blight": "Cotton___Bacterial_blight",
    "curl_virus": "Cotton___Curl_virus",
    "fussarium_wilt": "Cotton___Fusarium_wilt",
    "healthy": "Cotton___healthy",

    # Sugarcane
    "Rust": "Sugarcane___Rust",
    "Mosaic": "Sugarcane___Mosaic",
    "Yellow": "Sugarcane___Yellow",
    "Healthy": "Sugarcane___healthy",
    "RedRot": "Sugarcane___Red_rot"
}

# Function to copy images
def copy_images(src_folder, class_name, counter):
    dst_folder = os.path.join(merged_base, class_name)
    os.makedirs(dst_folder, exist_ok=True)
    for root, _, files in os.walk(src_folder):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                src_path = os.path.join(root, f)
                dst_path = os.path.join(dst_folder, f)
                shutil.copy(src_path, dst_path)
                counter[class_name] += 1

# Track image counts
counter = defaultdict(int)

# Process datasets
for dataset_name in datasets_to_add:
    dataset_path = os.path.join(source_base, dataset_name)
    if not os.path.exists(dataset_path):
        print(f"Skipping missing dataset: {dataset_name}")
        continue

    print(f"\nProcessing {dataset_name}...")
    for root, dirs, files in os.walk(dataset_path):
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            continue
        cls = os.path.basename(root)

        if dataset_name == "plantvillage-dataset":
            # Only include color images (skip grayscale/segmented folders)
            lower_cls = cls.lower()
            if "gray" in lower_cls or "grayscale" in lower_cls or "segmented" in lower_cls:
                continue
            unified_class = cls.replace(" ", "_")
        else:
            if cls not in manual_map:
                continue
            unified_class = manual_map[cls]

        copy_images(root, unified_class, counter)

# Summary
print("\nMerged dataset created at:", merged_base)
print("\nImage count per class:")
for cls_name, count in sorted(counter.items()):
    print(f"{cls_name}: {count}")



Processing plantvillage-dataset...

Processing rice-disease-dataset...

Processing cotton-leaf-disease-dataset...

Processing sugarcane-leaf-disease-dataset...

Merged dataset created at: /content/merged_dataset/train

Image count per class:
Apple___Apple_scab: 1890
Apple___Black_rot: 1863
Apple___Cedar_apple_rust: 825
Apple___healthy: 4935
Blueberry___healthy: 4506
Cherry_(including_sour)___Powdery_mildew: 3156
Cherry_(including_sour)___healthy: 2562
Corn_(maize)___Common_rust_: 3576
Corn_(maize)___Northern_Leaf_Blight: 2955
Corn_(maize)___healthy: 3486
Cotton___Bacterial_blight: 448
Cotton___Curl_virus: 417
Cotton___Fusarium_wilt: 419
Cotton___healthy: 425
Grape___Black_rot: 3540
Grape___Esca_(Black_Measles): 4150
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 3228
Grape___healthy: 1269
Orange___Haunglongbing_(Citrus_greening): 16521
Peach___Bacterial_spot: 6891
Peach___healthy: 1080
Pepper,_bell___Bacterial_spot: 2991
Pepper,_bell___healthy: 4434
Potato___Early_blight: 3000
Potato___L

**Removing underrepresented classes**

Classes such as Blueberry, Cherry, Orange, Peach, Raspberry, Squash, Soybean, and Strawberry are removed since they only contain healthy images and lack diseased samples.

In [ ]:

# Remove unwanted crops with insufficient diseased samples

import os
import shutil

dataset_path = "/content/merged_dataset/train"

# Crops to remove
crops_to_remove = [
    "Blueberry",
    "Cherry_(including_sour)",
    "Orange",
    "Peach",
    "Raspberry",
    "Squash",
    "Soybean",
    "Strawberry"
]

removed = []
for folder in os.listdir(dataset_path):
    for crop in crops_to_remove:
        if folder.startswith(crop):
            full_path = os.path.join(dataset_path, folder)
            shutil.rmtree(full_path)
            removed.append(folder)
            break

print("Removed classes:")
for r in removed:
    print(r)


Removed classes:
Blueberry___healthy
Cherry_(including_sour)___Powdery_mildew
Strawberry___Leaf_scorch
Orange___Haunglongbing_(Citrus_greening)
Strawberry___healthy
Cherry_(including_sour)___healthy
Squash___Powdery_mildew
Peach___Bacterial_spot
Raspberry___healthy
Soybean___healthy
Peach___healthy


**Verifying updated class distribution**

After removal and merging, this step checks the final image count per class to validate the modifications.

In [ ]:
import os
from collections import defaultdict

merged_base = "/content/merged_dataset/train"
updated_counter = defaultdict(int)

for class_folder in os.listdir(merged_base):
    class_path = os.path.join(merged_base, class_folder)
    if os.path.isdir(class_path):
        # Count files in each class folder
        image_count = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        updated_counter[class_folder] = image_count

print("\nUpdated image count per class in the merged dataset:")
for cls_name, count in sorted(updated_counter.items()):
    print(f"{cls_name}: {count}")


Updated image count per class in the merged dataset:
Apple___Apple_scab: 1260
Apple___Black_rot: 1242
Apple___Cedar_apple_rust: 550
Apple___healthy: 3290
Corn_(maize)___Common_rust_: 2384
Corn_(maize)___Northern_Leaf_Blight: 1970
Corn_(maize)___healthy: 2324
Cotton___Bacterial_blight: 448
Cotton___Curl_virus: 417
Cotton___Fusarium_wilt: 419
Cotton___healthy: 425
Grape___Black_rot: 2360
Grape___Esca_(Black_Measles): 2767
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 2152
Grape___healthy: 846
Pepper,_bell___Bacterial_spot: 1994
Pepper,_bell___healthy: 2956
Potato___Early_blight: 2000
Potato___Late_blight: 2000
Potato___healthy: 304
Rice___Bacterial_leaf_blight: 636
Rice___Brown_spot: 646
Rice___Leaf_blast: 634
Rice___Leaf_scald: 628
Rice___Sheath_blight: 632
Rice___healthy: 653
Sugarcane___Mosaic: 462
Sugarcane___Red_rot: 518
Sugarcane___Rust: 514
Sugarcane___Yellow: 505
Sugarcane___healthy: 522
Tomato___Bacterial_spot: 4254
Tomato___Early_blight: 2000
Tomato___Late_blight: 3818
Tomato___

**Balancing the dataset (upsampling & downsampling)**

Address dataset imbalance by:

**Upsampling:** Generate additional samples using augmentation (flipping, rotation, brightness changes).

**Downsampling:** Randomly reduce classes with excessive images.
This ensures uniform class sizes and prevents model bias.


In [ ]:
import os, random
from PIL import Image, ImageEnhance
from tqdm import tqdm
import shutil

input_base = "/content/merged_dataset/train"
output_base = "/content/balanced_dataset"
TARGET_COUNT = 1000

os.makedirs(output_base, exist_ok=True)

def augment_image(image_path):
    img = Image.open(image_path)
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    img = img.rotate(random.randint(-20, 20))
    enhancer = ImageEnhance.Brightness(img)
    img = enhancer.enhance(0.7 + random.random()*0.6)
    return img

for cls in os.listdir(input_base):
    cls_input = os.path.join(input_base, cls)
    cls_output = os.path.join(output_base, cls)
    os.makedirs(cls_output, exist_ok=True)
    images = [f for f in os.listdir(cls_input) if f.lower().endswith(('.jpg','.png','.jpeg'))]

    if len(images) >= TARGET_COUNT:
        # Downsample
        selected = random.sample(images, TARGET_COUNT)
        for img_name in selected:
            shutil.copy(os.path.join(cls_input, img_name), cls_output)
    else:
        # Copy all existing
        for img_name in images:
            shutil.copy(os.path.join(cls_input, img_name), cls_output)
        # Generate augmented images
        extra_needed = TARGET_COUNT - len(images)
        for i in range(extra_needed):
            img_name = random.choice(images)
            img_path = os.path.join(cls_input, img_name)
            aug_img = augment_image(img_path)
            aug_img.save(os.path.join(cls_output, f"aug_{i}_{img_name}"))


**Checking the results after Balancing**

All the classes are having same quantity of images


In [ ]:
import os

merged_base = "/content/balanced_dataset"

total_classes = len(os.listdir(merged_base))
print("Total classes after balancing:", total_classes)

for cls in sorted(os.listdir(merged_base)):
    cls_path = os.path.join(merged_base, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"{cls}: {count} images")


Total classes after balancing: 41
Apple___Apple_scab: 1000 images
Apple___Black_rot: 1000 images
Apple___Cedar_apple_rust: 1000 images
Apple___healthy: 1000 images
Corn_(maize)___Common_rust_: 1000 images
Corn_(maize)___Northern_Leaf_Blight: 1000 images
Corn_(maize)___healthy: 1000 images
Cotton___Bacterial_blight: 1000 images
Cotton___Curl_virus: 1000 images
Cotton___Fusarium_wilt: 1000 images
Cotton___healthy: 1000 images
Grape___Black_rot: 1000 images
Grape___Esca_(Black_Measles): 1000 images
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1000 images
Grape___healthy: 1000 images
Pepper,_bell___Bacterial_spot: 1000 images
Pepper,_bell___healthy: 1000 images
Potato___Early_blight: 1000 images
Potato___Late_blight: 1000 images
Potato___healthy: 1000 images
Rice___Bacterial_leaf_blight: 1000 images
Rice___Brown_spot: 1000 images
Rice___Leaf_blast: 1000 images
Rice___Leaf_scald: 1000 images
Rice___Sheath_blight: 1000 images
Rice___healthy: 1000 images
Sugarcane___Mosaic: 1000 images
Sugarca

**Splitting the dataset into Train, Validation, and Test sets**

Divide the balanced dataset into training (70%), validation (20%), and testing (10%) subsets to ensure reliable model training and evaluation.



In [ ]:
import os
import shutil
import random
from tqdm import tqdm

source_dir = "/content/balanced_dataset"
output_base = "/content/final_dataset"

splits = {
    "train": 0.7,
    "val": 0.2,
    "test": 0.1
}

if os.path.exists(output_base):
    shutil.rmtree(output_base)
os.makedirs(output_base)

for split in splits:
    os.makedirs(os.path.join(output_base, split), exist_ok=True)

for cls in tqdm(os.listdir(source_dir), desc="Splitting classes"):
    cls_path = os.path.join(source_dir, cls)
    if not os.path.isdir(cls_path):
        continue

    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(images)

    total = len(images)
    train_end = int(splits['train'] * total)
    val_end = train_end + int(splits['val'] * total)

    split_files = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, files in split_files.items():
        dest_dir = os.path.join(output_base, split, cls)
        os.makedirs(dest_dir, exist_ok=True)
        for f in files:
            shutil.copy(os.path.join(cls_path, f), os.path.join(dest_dir, f))

print("Split complete! Final dataset in:", output_base)


Splitting classes: 100%|██████████| 41/41 [00:12<00:00,  3.18it/s]

Split complete! Final dataset in: /content/final_dataset


**Converting the dataset into Zip format**

Create a compressed .zip archive of the final dataset for easier storage, transfer, and reusability.

In [ ]:
import shutil
import os

output_base = "/content/final_dataset"
zip_filename = "/content/final_dataset.zip"

# Create a zip archive
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', output_base)

print(f"Created zip file: {zip_filename}")

Created zip file: /content/final_dataset.zip


 **Uploading dataset to Google Drive**

Save the zipped dataset to Google Drive for backup and external access.

In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the zip file and the destination in Google Drive
zip_file_path = "/content/final_dataset.zip"
drive_destination_path = "/content/drive/My Drive/final_dataset.zip" # You can change this path

# Copy the zip file to Google Drive
if os.path.exists(zip_file_path):
    shutil.copy(zip_file_path, drive_destination_path)
    print(f"Zip file copied to Google Drive at: {drive_destination_path}")
else:
    print(f"Error: Zip file not found at {zip_file_path}")

 **Downloading dataset in zip format**

Provide direct download of the dataset archive (final_dataset.zip) for local use outside of Colab.

In [ ]:
from google.colab import files

zip_filename = "/content/final_dataset.zip"

# Download the zip file
print(f"Downloading {zip_filename}...")
files.download(zip_filename)